# Check that the selected TCGA slides are likely FFPE H&E

In [1]:
benchmark_id = "tcga_benchmark_1"

In [3]:
import re
import sys
import pandas as pd

# Path to the benchmark's slide manifest:
path = f"/path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/benchmarks/{benchmark_id}/shared/cohort/slide_manifest.csv"

# slide_manifest.csv
df = pd.read_csv(path, dtype=str, keep_default_na=False)  # keep blanks as "" not NaN
n = len(df)

required = ["experimental_strategy", "data_type", "data_format",
           "sample_type", "slide_file_name", "case_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    sys.exit(f"ERROR: slide_manifest.csv is missing columns: {missing}")

print(f"Loaded {n} selected slides from {path}\n")

PRIMARY_SAMPLE_TYPES = {"Primary Tumor", "Primary Solid Tumor"}

checks = {
    "experimental_strategy == 'Diagnostic Slide'":
        df["experimental_strategy"].str.strip().eq("Diagnostic Slide"),
    "data_type == 'Slide Image'":
        df["data_type"].str.strip().eq("Slide Image"),
    "data_format == 'SVS'":
        df["data_format"].str.strip().str.upper().eq("SVS"),
    "sample_type is primary tumor":
        df["sample_type"].str.strip().isin(PRIMARY_SAMPLE_TYPES),
    # Canonical TCGA diagnostic (FFPE) barcode token, e.g. ...-01Z-00-DX2....svs
    "slide_file_name has a -DX# token":
        df["slide_file_name"].str.contains(r"-DX\d+", case=False, regex=True),
    "slide_file_name ends with .svs":
        df["slide_file_name"].str.strip().str.lower().str.endswith(".svs"),
}

combined = pd.Series(True, index=df.index)
all_ok = True
for label, mask in checks.items():
    mask = mask.fillna(False)
    combined &= mask
    n_fail = int((~mask).sum())
    status = "OK" if n_fail == 0 else "FAIL"
    print(f"[{status}] {label}: {int(mask.sum())}/{n} pass"
          + ("" if n_fail == 0 else f"   ({n_fail} fail)"))
    if n_fail:
        all_ok = False
        for _, r in df.loc[~mask, ["case_id", "slide_file_name"]].head(10).iterrows():
            print(f"          - {r['case_id']}  {r['slide_file_name']}")

print()
if all_ok:
    print(f"All {n} slides match the H&E-FFPE diagnostic pattern on every check.")
else:
    n_bad = int((~combined).sum())
    print(f"{n_bad} of {n} slides fail at least one check (offending rows listed above).")

Loaded 866 selected slides from /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/benchmarks/tcga_benchmark_1/shared/cohort/slide_manifest.csv

[OK] experimental_strategy == 'Diagnostic Slide': 866/866 pass
[OK] data_type == 'Slide Image': 866/866 pass
[OK] data_format == 'SVS': 866/866 pass
[OK] sample_type is primary tumor: 866/866 pass
[OK] slide_file_name has a -DX# token: 866/866 pass
[OK] slide_file_name ends with .svs: 866/866 pass

All 866 slides match the H&E-FFPE diagnostic pattern on every check.
